# Tetris RL — CC421

Función de valor lineal `V(σ)=wᵀφ(σ)` para Tetris: baselines (aleatorio, heurístico) frente a los métodos que aprenden **TD** y **CEM**.

## 1. Setup

In [ ]:
import os

REPO_URL = 'https://github.com/FixerCoda/CC421_Proyecto_PC5.git'
REPO_DIR = 'CC421_Proyecto_PC5'

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
if os.path.basename(os.getcwd()) != REPO_DIR:
    os.chdir(REPO_DIR)

!pip install -q -e .
!pip install -q pytest            # pytest no está en las deps del paquete
print('Directorio de trabajo:', os.getcwd())

## 2. Tests

In [ ]:
!pytest -q

## 3. Generación de datos

Partidas con `RandomAgent`: una fila por colocación en `data/raw/`.

In [ ]:
!python -m tetris_rl.data.generate --episodes 100 --seed 0

In [ ]:
from pathlib import Path
import pandas as pd

csv = sorted(Path('data/raw').glob('RandomAgent_*'))[-1]
df = pd.read_csv(csv)
print(csv, '->', len(df), 'transiciones')
df.head()

### EDA de las trayectorias

In [ ]:
import matplotlib.pyplot as plt

per_ep = df.groupby('episode').agg(
    steps=('step', 'size'),
    lines=('lines_cleared', 'sum'),
    reward=('reward', 'sum'),
)
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].hist(per_ep['steps'], bins=20); ax[0].set_title('Duración de episodios (colocaciones)')
ax[1].hist(per_ep['lines'], bins=20); ax[1].set_title('Líneas por episodio')
ax[2].hist(per_ep['reward'], bins=20); ax[2].set_title('Recompensa por episodio')
plt.tight_layout(); plt.show()
per_ep.describe()

Con `RandomAgent` casi no se completan líneas y los episodios son cortos: recompensa dispersa, lo que motiva usar agentes que aprenden.

## 4. Comparación de agentes

`evaluate.py` juega con las mismas semillas y un tope de colocaciones, usando los pesos de `models/`. Reporta líneas (media/desv/máx) y la norma ‖w‖ de cada modelo.

In [ ]:
!python evaluate.py --episodes 20 --max-steps 500

## 5. Entrenamiento

Corrida corta del CEM (los pesos finales versionados usan la config completa). Otros entrypoints: `train_td.py` y `sweep.py`.

In [ ]:
!python train_cem.py --generations 8 --population 20 --episodes 3 --max-steps 300 --seed 0

## 6. Juego visual

`python play_model.py` reproduce una partida del agente en modo texto (mejor en local; en Colab no hay render interactivo fluido).